In [1]:
import pandas as pd
import pypsa

In [23]:

n = pypsa.Network()

# 1. Add buses
for bus in ["A", "B", "C", "D"]:
    n.add("Bus", bus, v_nom=1.0)

# 2. Add transmission lines
line_data = [
    ("AB", "A", "B", 10),
    ("BC", "B", "C", 9),
    ("CD", "C", "D", 11),
    ("AD", "A", "D", 10),
    ("BD", "B", "D", 6),
]

for name, bus0, bus1, capacity in line_data:
    n.add(
        "Line",
        name,
        bus0=bus0,
        bus1=bus1,
        x=1.0,
        r=0.0,
        s_nom=capacity
    )

# 3. Add generators
generator_data = [
    ("G1", "A", 10, 20),
    ("G2", "A", 8, 35),
    ("G3", "B", 12, 65),
    ("G4", "D", 16, 90),
]

for name, bus, capacity, cost in generator_data:
    n.add(
        "Generator",
        name,
        bus=bus,
        p_nom=capacity,
        marginal_cost=cost
    )

# 4. Add loads
n.add("Load", "Load_C", bus="C", p_set=4.99)
n.add("Load", "Load_D", bus="D", p_set=11)


In [24]:

n_normal = n.copy()

n.sanitize()

status, condition = n_normal.optimize(
    solver_name="highs",
    include_objective_constant=False
)

print("Status:", status, condition)

print("\nGenerator dispatch:")
print(n_normal.generators_t.p.iloc[0])

print("\nTotal cost:")
print(n_normal.objective)

print("\nLine flows:")
print(n_normal.lines_t.p0.iloc[0])

print("\nNodal prices:")
print(n_normal.buses_t.marginal_price.iloc[0])


INFO:pypsa.consistency:Sanitizing network...
INFO:pypsa.components._types.carriers:Adding 1 missing carriers: ['AC']
INFO:pypsa.components._types.carriers:Assigned colors to 1 carriers using 'tab10' palette.
INFO:pypsa.consistency:Network sanitization complete.
Index(['A', 'B', 'C', 'D'], dtype='object', name='name')
Index(['AB', 'BC', 'CD', 'AD', 'BD'], dtype='object', name='name')
Index(['AB', 'BC', 'CD', 'AD', 'BD'], dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.02s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 9 primals, 24 duals
Objective: 4.10e+02
Solver: highs
Runtime: 0.00s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper, Kirchhoff-Voltage-Law were not assigned to the network

Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-r50k5rhx has 24 rows; 9 cols; 38 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+05]
  Cost    [2e+01, 9e+01]
  Bound   [0e+00, 0e+00]
  RHS     [5e+00, 2e+01]
Presolving model
4 rows, 7 cols, 13 nonzeros 0s
Dependent equations search running on 4 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
4 rows, 7 cols, 13 nonzeros 0s
Presolve reductions: rows 4(-20); columns 7(-2); nonzeros 13(-25) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0    -2.6955412123e-04 Pr: 4(2020.97) 0.0s
          4     4.0965000000e+02 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-r50k5rhx
Model

In [25]:

n_secure = n.copy()

status, condition = (
    n_secure.optimize.optimize_security_constrained(
        branch_outages=["BD", "BC"],
        solver_name="highs"
    )
)

print("Status:", status, condition)

print("\nSecurity-constrained dispatch:")
print(n_secure.generators_t.p.iloc[0])

print("\nTotal cost:")
print(n_secure.objective)

print("\nNodal prices:")
print(n_secure.buses_t.marginal_price.iloc[0])


Index(['AB', 'BC', 'CD', 'AD', 'BD'], dtype='object', name='name')
/Users/davidreynolds/.local/share/uv/python/cpython-3.14.7-macos-aarch64-none/lib/python3.14/contextlib.py:85: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  return func(*args, **kwds)
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.02s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 9 primals, 44 duals
Objective: 4.64e+02
Solver: highs
Runtime: 0.00s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper, Kirchhoff-Voltag

Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-rmfffclt has 44 rows; 9 cols; 70 nonzeros
Coefficient ranges:
  Matrix  [3e-01, 1e+05]
  Cost    [2e+01, 9e+01]
  Bound   [0e+00, 0e+00]
  RHS     [5e+00, 2e+01]
Presolving model
12 rows, 7 cols, 29 nonzeros 0s
Dependent equations search running on 3 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
7 rows, 6 cols, 18 nonzeros 0s
Presolve reductions: rows 7(-37); columns 6(-3); nonzeros 18(-52) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0    -2.2782949033e-06 Pr: 5(14.4444) 0.0s
          5     4.6428333333e+02 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-rmfffclt
Mode

In [15]:
# Copy the security-constrained network
check = n.copy()

# Give it the security-constrained dispatch
check.generators_t.p_set = n_secure.generators_t.p.copy()

# Simulate the two outages
outage_flows = check.lpf_contingency(
    snapshots=[check.snapshots[0]],
    branch_outages=["BD", "BC"]
)

print(outage_flows)

INFO:pypsa.network.power_flow:Performing linear load-flow on AC sub-network <pypsa.networks.SubNetwork object at 0x1348f6210> for snapshot(s) Index(['now'], dtype='str', name='snapshot')


                base    (Line, BD)  (Line, BC)
component name                                
Line      AB    6.25  5.000000e+00         5.0
          BC    3.75  5.000000e+00         0.0
          CD   -1.25 -2.220446e-16        -5.0
          AD    8.75  1.000000e+01        10.0
          BD    2.50  0.000000e+00         5.0


In [14]:
n_secure.buses_t.marginal_price

name,A,B,C,D
snapshot,,,,
now,35.0,62.5,90.0,90.0


In [26]:
#.9/.01

(464.2833333333334 - 465)/.01


-71.66666666665833